# 05 — Tracking e deslocamento aparente

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USUARIO/projeto-plumas/blob/main/05_tracking/05_tracking.ipynb)

**Objetivo:** associar as detecções entre quadros consecutivos e medir como posição e forma mudam, sempre em pixels.

**Escopo da professora:** seção 8 (associação por centroide/IoU/consistência temporal; registrar períodos sem detecção; separar deslocamento aparente, vento e trajetória prevista) e pergunta 3.

**Lê:** `04_sam_atributos/atributos.csv` e `01_dados/frames.csv`.

**Grava:** `trajetoria.csv`, `sem_deteccao.csv`, figuras; opcionalmente `fluxo_optico.csv`.

**GPU:** não


In [ ]:
# === Preparo (igual em todos os notebooks) ===
import os, sys, subprocess

REPO = "USUARIO/projeto-plumas"      # troque pelo seu (veja README)
REPO_DIR = "/content/projeto-plumas"

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    try:                                   # repositório privado: secret GITHUB_TOKEN
        from google.colab import userdata
        _tok = userdata.get("GITHUB_TOKEN")
    except Exception:
        _tok = None
    _url = f"https://{_tok}@github.com/{REPO}.git" if _tok else f"https://github.com/{REPO}.git"
    if os.path.isdir(REPO_DIR):
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], capture_output=True)
    else:
        _r = subprocess.run(["git", "clone", "-q", _url, REPO_DIR], capture_output=True, text=True)
        if _r.returncode != 0:
            raise RuntimeError("Falha no clone. Repositório privado? Configure o secret GITHUB_TOKEN (veja README).")
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:                                      # execução local: sobe até a raiz do repo
    _p = os.getcwd()
    while not os.path.exists(os.path.join(_p, "config.yaml")) and os.path.dirname(_p) != _p:
        _p = os.path.dirname(_p)
    os.chdir(_p)
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

from pathlib import Path
from plumas.config import load_config, stage_dir, set_seed, log_experiment
cfg = load_config()
set_seed(cfg["seed"])
STAGE = "05_tracking"
D = stage_dir(cfg, STAGE)                  # pasta desta etapa no Drive
print("Etapa:", STAGE, "| dados em:", D)


## 1. Associar detecções entre quadros
Associação gulosa por IoU entre quadros consecutivos. Se o intervalo entre dois quadros com detecção passar de
`tracking.max_gap_s`, a trajetória é **interrompida**: não se trata um intervalo longo sem observação como se a trajetória fosse conhecida.

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from plumas import tracking as T

tcfg = cfg["tracking"]
att = pd.read_csv(stage_dir(cfg, "04_sam_atributos") / "atributos.csv")
frames = pd.read_csv(stage_dir(cfg, "01_dados") / "frames.csv")
df = att.merge(frames, on="frame", how="left").sort_values(["sequence", "t_s", "det_id"]).reset_index(drop=True)

rows, next_id = [], 0
for seq, g in df.groupby("sequence"):
    prev = None
    for fr, gf in g.groupby("frame", sort=False):
        t = float(gf["t_s"].iloc[0])
        boxes = gf[["x1", "y1", "x2", "y2"]].values.tolist()
        ids = [None] * len(boxes)
        if prev is not None and (t - prev["t"]) <= tcfg["max_gap_s"]:
            for i, j in T.associate(prev["boxes"], boxes, tcfg["iou_threshold"]):
                ids[j] = prev["ids"][i]
        for j in range(len(ids)):
            if ids[j] is None:
                ids[j] = next_id; next_id += 1
        for (_, r), tid in zip(gf.iterrows(), ids):
            rows.append({**r.to_dict(), "track_id": tid})
        prev = {"t": t, "boxes": boxes, "ids": ids}

tr = pd.DataFrame(rows).sort_values(["track_id", "t_s"]).reset_index(drop=True)
print(tr.groupby("track_id").size().describe())

## 2. Deslocamento aparente Δc
Δc = c(t+1) − c(t), com c = (u, v) em coordenadas de imagem (u para a direita, **v para baixo**). É uma mudança em **pixels**,
não uma velocidade física. `speed_px_s` é só pixels por segundo na imagem.

Atenção: o centroide também muda quando a pluma **cresce ou muda de forma**, sem que ela "ande". Compare com o fluxo óptico (seção 4).

In [ ]:
g = tr.groupby("track_id")
tr["dx_px"] = g["cx"].diff()
tr["dy_px"] = g["cy"].diff()
tr["dt_s"] = g["t_s"].diff()
tr["speed_px_s"] = np.hypot(tr["dx_px"], tr["dy_px"]) / tr["dt_s"]
tr["d_area_px"] = g["area_px"].diff()
tr["d_orient_deg"] = g["orientation_deg"].diff()
tr.to_csv(D / "trajetoria.csv", index=False)
tr[["frame", "track_id", "t_s", "cx", "cy", "dx_px", "dy_px", "speed_px_s", "area_px", "orientation_deg"]].head(10)

## 3. Períodos sem detecção
Registre onde a pluma **não** foi detectada. Esses intervalos entram no relatório como lacunas, sem interpolação.

In [ ]:
detected = set(df["frame"])
sem = frames[~frames["frame"].isin(detected)].copy()
sem.to_csv(D / "sem_deteccao.csv", index=False)

gaps = {}
for seq, g_ in df.groupby("sequence"):
    t_det = sorted(g_["t_s"].unique())
    gaps[seq] = [(t_det[k], t_det[k + 1]) for k in T.find_gaps(t_det, tcfg["max_gap_s"])]
print(f"Quadros sem detecção: {len(sem)} de {len(frames)} ({100*len(sem)/len(frames):.1f}%)")
print("Lacunas maiores que max_gap_s (início, fim em s):", gaps)

fig, ax = plt.subplots(figsize=(12, 2.5))
for seq, g_ in frames.groupby("sequence"):
    ax.scatter(g_["t_s"], [seq] * len(g_), c=["tab:green" if f in detected else "lightgray" for f in g_["frame"]], s=12)
ax.set_xlabel("tempo (s)"); ax.set_title("verde = pluma detectada, cinza = sem detecção"); plt.tight_layout(); plt.show()

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(18, 4.5))
for tid, g_ in tr.groupby("track_id"):
    if len(g_) < 3:
        continue
    axs[0].plot(g_["cx"], g_["cy"], "-o", ms=3, label=f"track {tid}")
    axs[1].plot(g_["t_s"], g_["area_px"], label=f"track {tid}")
    axs[2].plot(g_["t_s"], g_["orientation_deg"], label=f"track {tid}")
axs[0].invert_yaxis(); axs[0].set_title("centroide na imagem (v para baixo)"); axs[0].set_xlabel("u (px)"); axs[0].set_ylabel("v (px)")
axs[1].set_title("área aparente (px)"); axs[1].set_xlabel("t (s)")
axs[2].set_title("orientação (graus)"); axs[2].set_xlabel("t (s)")
for a in axs: a.grid(alpha=.3)
axs[0].legend(fontsize=7); plt.tight_layout()
plt.savefig(D / "trajetoria_resumo.png", dpi=150); plt.show()

## 4. (Opcional) Fluxo óptico dentro da máscara
Mede o movimento **dos pixels** da pluma, independente do crescimento da forma. Farneback funciona melhor com deslocamentos pequenos:
com quadros extraídos a cada 0,5 s o movimento pode ser grande demais; se o resultado ficar ruim, extraia quadros mais próximos (diminua `video.frame_step_s`).
É igualmente movimento aparente em pixels.

In [ ]:
import cv2
frames_dir = stage_dir(cfg, "01_dados") / "frames"
mask_dir = stage_dir(cfg, "04_sam_atributos") / "mascaras"
flow_rows = []
for seq, g_ in df.groupby("sequence"):
    fr_list = frames[frames["sequence"] == seq].sort_values("t_s")["frame"].tolist()
    mask_by_frame = df[df["sequence"] == seq].groupby("frame")["mask_file"].apply(list).to_dict()
    for a, b in zip(fr_list[:-1], fr_list[1:]):
        if a not in mask_by_frame or b not in mask_by_frame:
            continue
        ga = cv2.imread(str(frames_dir / a), 0); gb = cv2.imread(str(frames_dir / b), 0)
        flow = cv2.calcOpticalFlowFarneback(ga, gb, None, 0.5, 3, 25, 3, 5, 1.2, 0)
        m = np.zeros_like(ga, bool)
        for f in mask_by_frame[a]:
            m |= cv2.imread(str(mask_dir / f), 0) > 0
        if m.sum() == 0:
            continue
        t_a = float(frames.loc[frames["frame"] == a, "t_s"].iloc[0]); t_b = float(frames.loc[frames["frame"] == b, "t_s"].iloc[0])
        flow_rows.append({"sequence": seq, "frame_a": a, "frame_b": b, "dt_s": t_b - t_a,
                          "flow_dx_px": float(flow[..., 0][m].mean()), "flow_dy_px": float(flow[..., 1][m].mean())})
flow_df = pd.DataFrame(flow_rows)
flow_df.to_csv(D / "fluxo_optico.csv", index=False)
flow_df.head()

In [ ]:
log_experiment(cfg, STAGE, {"iou_threshold": tcfg["iou_threshold"], "max_gap_s": tcfg["max_gap_s"],
                            "n_tracks": int(tr["track_id"].nunique()), "frac_sem_deteccao": float(len(sem) / len(frames)),
                            "decisao": "lacunas registradas sem interpolação"})